# Azure Container Apps as a sandbox for Deep Agents

An agent that writes code needs somewhere to run it. Running it in your own
process is not an option once a model is choosing the code, so the execution has
to happen somewhere isolated, disposable, and outside your trust boundary.

Azure Container Apps offers **two separate products** for this, and
`langchain-azure-container-apps` wraps both:

| | Dynamic sessions | Sandboxes |
|---|---|---|
| ARM resource | `Microsoft.App/sessionPools` | `Microsoft.App/sandboxGroups` |
| Access | HTTP via one pool endpoint | per-sandbox data plane SDK |
| State | ephemeral, destroyed after cooldown | stateful: stop, resume, snapshot |
| Persistent storage | none | volumes (Azure Blob, Data Disk) |
| Networking | basic isolation | egress policies, VNet, port management |
| Startup | pre-warmed, ~instant | provisioned per sandbox, ~10s |
| Best for | one-shot LLM-generated code | long-running agent workspaces |

They are **not tiers of one product**. Different resource types, different data
planes, different lifecycles. This notebook covers dynamic sessions first
(Part 1), then sandboxes (Part 2), and closes with how to choose (Part 3).

## What you will build

1. A LangChain agent whose only tool is a Python REPL running in Azure.
2. A Deep Agent whose entire filesystem *is* a remote shell session.
3. A Deep Agent that owns a stateful sandbox — installs packages into it,
   survives a stop/resume, and gets torn down at the end.

## Before you start

**Azure resources.** This notebook assumes these already exist:

- a **Python**-typed session pool and a **Shell**-typed session pool
- a **sandbox group**

**Role assignments** on those resources, for the identity you `az login` with:

| Resource | Role |
|---|---|
| Session pools | `Azure ContainerApps Session Executor` |
| Sandbox group | `Container Apps SandboxGroup Data Owner` |

**Authentication** is `DefaultAzureCredential`, which shells out to the Azure
CLI. `az login` is enough — no service principal, no secret on disk. If you keep
several CLI profiles, select the right one *before launching Jupyter*:

```bash
export AZURE_CONFIG_DIR=~/.config/azure/<profile>
az account show --query "{sub:name, user:user.name}" -o tsv
```

**Configuration** lives in `.env` (copy `.env.example`). It holds resource
coordinates only. Your model key is read from the shell environment and is never
written to disk:

```bash
export OPENAI_API_KEY=...
jupyter lab
```

## Installation

The package is not on PyPI yet, so `pyproject.toml` pulls it from the branch:

```toml
[project]
dependencies = ["langchain-azure-container-apps[dynamic-sessions,sandboxes]"]

[tool.uv.sources]
langchain-azure-container-apps = { git = "https://github.com/langchain-ai/langchain-azure.git", branch = "feat/aca-06-deprecate-dynamic-sessions", subdirectory = "libs/azure-container-apps" }
```

Then `uv sync`. Once it ships, that becomes a plain
`pip install "langchain-azure-container-apps[dynamic-sessions,sandboxes]"`.

The two extras are independent — install only the one you need. They gate
*dependencies*, not modules: every module is in the wheel, and each subpackage
raises an import error naming the extra it needs.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

PYTHON_POOL = os.environ["AZURE_DYNAMIC_SESSIONS_POOL_MANAGEMENT_ENDPOINT"]
SHELL_POOL = os.environ["AZURE_DYNAMIC_SESSIONS_SHELL_POOL_MANAGEMENT_ENDPOINT"]

SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["AZURE_CONTAINER_APPS_RESOURCE_GROUP"]
SANDBOX_GROUP = os.environ["AZURE_CONTAINER_APPS_SANDBOX_GROUP"]
REGION = os.environ["AZURE_CONTAINER_APPS_REGION"]

# Print the pool *names* rather than the endpoints: the full URL embeds the
# subscription id, and this notebook is meant to be shareable.
print("python pool :", PYTHON_POOL.rsplit("/", 1)[-1])
print("shell pool  :", SHELL_POOL.rsplit("/", 1)[-1])
print("sandbox grp :", SANDBOX_GROUP)
print("region      :", REGION)

python pool : sp-python
shell pool  : sp-shell
sandbox grp : sandbox-group-demo
region      : westus2


### The model

Any tool-calling chat model works. This notebook uses OpenAI via
`langchain-openai`, which reads `OPENAI_API_KEY` (and `OPENAI_BASE_URL`, if you
route through a gateway) straight from the environment.

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

# Confirm the key resolves before anything expensive runs.
print(model.invoke("Reply with exactly: ready").content)

ready


### A note on `LangChainBetaWarning`

Both Deep Agents backends are marked `@beta`, so importing them emits a warning.
That is expected and worth reading rather than suppressing: deepagents is pre-1.0
and ACA sandboxes is an Early Access service, so these interfaces can still
change. We leave the warnings visible throughout.

---

# Part 1 — Dynamic Sessions

A **session pool** (`Microsoft.App/sessionPools`) is a managed fleet of
pre-warmed, identical, isolated interpreters. You do not create or destroy
sessions; you address one by an arbitrary `session_id` string against the pool's
endpoint, and Azure allocates it on first use and reclaims it after a cooldown.

That model has a sharp consequence: **a session is ephemeral**. Everything in it
disappears once it goes idle. There is no stop, no resume, no snapshot, no
volume. In exchange you get near-instant startup and no lifecycle to manage.

Pools are **typed** at creation, and the type is not cosmetic:

- a **PythonLTS** pool runs Python code and returns a structured result
- a **Shell** pool runs shell commands

A tool aimed at the wrong pool type will fail. We use both below.

## 1.1 `SessionsPythonREPLTool`

The simplest surface in the package: hand it Python source, get back the
session's `result`, `stdout`, and `stderr` as JSON.

In [3]:
from langchain_azure_container_apps.dynamic_sessions import SessionsPythonREPLTool

repl = SessionsPythonREPLTool(pool_management_endpoint=PYTHON_POOL)

print(repl.invoke("print(sum(range(10)))"))

{
  "result": "",
  "stdout": "45\n",
  "stderr": ""
}


Note what came back: `result` is empty and `stdout` has the text. The Python pool
distinguishes the **value of the last expression** from what the code *printed*.
Ask for an expression instead and it lands in `result`:

In [4]:
print(repl.invoke("sum(range(10))"))

{
  "result": 45,
  "stdout": "",
  "stderr": ""
}


Sessions are keyed by `session_id`. The tool generates one per instance, so two
tools are two isolated interpreters — and state does persist *within* one session
for as long as it stays warm:

In [5]:
repl.invoke("x = 1234")
print("same tool, later call :", repl.invoke("x"))

other = SessionsPythonREPLTool(pool_management_endpoint=PYTHON_POOL)
print("a different session   :", other.invoke("'x' in dir()"))

same tool, later call : {
  "result": 1234,
  "stdout": "",
  "stderr": ""
}


a different session   : {
  "result": false,
  "stdout": "",
  "stderr": ""
}


Two things follow from this, and both matter in production:

- **Give each end user their own `session_id`.** Sharing one across users leaks
  state between them. Pass `session_id=` explicitly and derive it from your own
  conversation or user identifier.
- **Do not rely on that state.** It survives calls, not the cooldown. Treat the
  session as a scratchpad that may be empty on the next call.

## 1.2 Giving the tool to an agent

`SessionsPythonREPLTool` is an ordinary LangChain tool, so it drops into
`create_agent` with nothing special. This is the classic "let the model do
arithmetic it would otherwise hallucinate" pattern — except the code runs in
Azure, not here.

In [6]:
from langchain.agents import create_agent

calc = create_agent(
    model=model,
    tools=[SessionsPythonREPLTool(pool_management_endpoint=PYTHON_POOL)],
    system_prompt=(
        "You are a careful analyst. Whenever a question requires computation, "
        "write Python and run it with the REPL tool rather than doing arithmetic "
        "yourself. Always print the answer."
    ),
)

result = calc.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What are the last four digits of 7**291, and how many "
                    "digits does it have in total?"
                ),
            }
        ]
    }
)
print(result["messages"][-1].content)

The last four digits are **0743**.

It has **246 digits** in total.


The interesting part is the trajectory, not the answer. Let's see what the model
actually chose to execute in Azure:

In [7]:
for message in result["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        print(f"--- {call['name']} ---")
        print(call["args"].get("python_code", call["args"]))

--- Python_REPL ---
n = 7**291
print(n % 10000)
print(len(str(n)))


## 1.3 `SessionsBashTool`

The same idea against a **Shell**-typed pool. The response shape differs: an
`exitCode` rather than a `result`, since shells report success numerically.

In [8]:
from langchain_azure_container_apps.dynamic_sessions import SessionsBashTool

bash = SessionsBashTool(pool_management_endpoint=SHELL_POOL)

print(bash.invoke("uname -s; echo $((6 * 7))"))

{
  "stdout": "Linux\n42\n",
  "stderr": "",
  "exitCode": 0
}


## 1.4 `SessionsBashBackend` — a Deep Agents backend

Tools give a model a place to *run* code. A Deep Agents **backend** goes further:
it becomes the agent's whole filesystem. `ls`, `read`, `write`, `edit`, `glob`,
`grep` and `execute` are all served by the remote session, so the agent's notion
of "the files" is the session's disk, not yours.

`SessionsBashBackend` implements that protocol against a Shell pool. Its file
operations are **bash-native** rather than the `python3 -c` wrappers the generic
`BaseSandbox` uses — deliberately, because a shell-only pool image may have no
Python to run them with.

You can drive it directly, which is the clearest way to see what the agent sees:

In [9]:
from langchain_azure_container_apps.dynamic_sessions.backends import (
    SessionsBashBackend,
)

backend = SessionsBashBackend(pool_management_endpoint=SHELL_POOL)

print("write :", backend.write("/tmp/notes.txt", "alpha\nbeta\ngamma").error or "ok")

read = backend.read("/tmp/notes.txt")
print("read  :", read.file_data["content"])
print("lines :", read.total_lines)

print("exec  :", backend.execute("wc -l < /tmp/notes.txt").output.strip())

/var/folders/0w/mqr_p79n6n78wrvzd45x2jc40000gn/T/ipykernel_6182/2787238380.py:5: LangChainBetaWarning: `SessionsBashBackend` is in public preview. Its API is not stable and may change in future versions.
  backend = SessionsBashBackend(pool_management_endpoint=SHELL_POOL)


write : ok


read  : alpha
beta
gamma
lines : 3
exec  : 2


These return **typed results**, not strings — `ReadResult`, `LsResult`,
`ExecuteResponse` — which is what lets Deep Agents paginate a large file instead
of dropping it whole into the context window:

In [10]:
backend.write("/tmp/many.txt", "\n".join(f"line {i}" for i in range(1, 101)))

window = backend.read("/tmp/many.txt", offset=10, limit=3)
print("content     :", repr(window.file_data["content"]))
print("total_lines :", window.total_lines)
print("next_offset :", window.next_offset)

content     : 'line 11\nline 12\nline 13'
total_lines : 100
next_offset : 13


### Now hand it to an agent

`create_deep_agent(backend=...)` is the whole integration. The agent gets its
filesystem tools automatically, and every one of them lands in the Azure session.

In [11]:
from deepagents import create_deep_agent

sessions_agent = create_deep_agent(
    model=model,
    backend=SessionsBashBackend(pool_management_endpoint=SHELL_POOL),
    system_prompt=(
        "You are a coding assistant working in a Linux shell sandbox. "
        "Write files with the file tools, run them with execute, and report "
        "what actually happened. Keep command output small."
    ),
)

response = sessions_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Write /tmp/fib.py that prints the first 15 Fibonacci "
                    "numbers on one line, space separated. Run it and tell me "
                    "the output."
                ),
            }
        ]
    }
)
print(response["messages"][-1].content)

I wrote `/tmp/fib.py` and ran it.

Output:
`0 1 1 2 3 5 8 13 21 34 55 89 144 233 377`


Every file tool the agent called went to the session pool. Here is the trajectory:

In [12]:
for message in response["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        args = call["args"]
        detail = args.get("file_path") or args.get("command") or ""
        print(f"{call['name']:12} {str(detail)[:70]}")

write_file   /tmp/fib.py
execute      python3 /tmp/fib.py


## 1.5 Limits worth knowing before you ship

These are properties of the session data plane, not of the wrapper. The backend
works around what it can and tells you about the rest.

**Output is capped at 4,096 bytes** per stream, silently, by the service.
`read` pages beneath the cap so any file within `max_output_bytes` comes back
intact. `ls`, `glob` and `grep` detect the cap and say so rather than passing off
a truncated answer as a complete one. `execute` sets `truncated=True` on a stream
that arrives exactly at the cap — a heuristic, and the only signal available:

In [13]:
big = backend.execute("head -c 5000 /dev/zero | tr '\\0' 'x'")
print("bytes returned :", len(big.output))
print("truncated flag :", big.truncated)

bytes returned : 4096
truncated flag : True


**The final line of output is intermittently dropped** by the data plane
(measured at 1.6–4%). `ls`, `read`, `glob` and `grep` detect this and retry.
**`execute` deliberately does not** — it runs your command verbatim, and wrapping
it would change the exit status it returns. If you parse the last line of
`execute` output, terminate your command with a marker of your own and re-run
when the marker is missing.

**`write` refuses to overwrite**, matching the shared sandbox test suite. The
error tells the model to use `edit` or delete first, which is more useful to an
agent than a silent clobber:

In [14]:
print(backend.write("/tmp/notes.txt", "different content").error)

File '/tmp/notes.txt' already exists. Use edit to modify it, or delete it first.


**Large payloads must not ride inside a command.** `write` content and `edit`
strings travel in a single shell command, which Linux caps at 128 KiB per string;
anything above ~90 KB is refused with an explanatory error. `edit` budgets
`old_string` and `new_string` *together*, since both are in that one command.

**`upload_files` / `download_files` use the session file API**, which is a flat
store rooted at `/mnt/data`. Only `/mnt/data/<name>` is storable — any other path
is rejected with `invalid_path` rather than being silently written elsewhere.
`write` has no such limit, because it goes through the shell:

In [15]:
uploaded = backend.upload_files([("/mnt/data/blob.bin", bytes(range(256)))])
print("to /mnt/data :", uploaded[0].error or "ok")

rejected = backend.upload_files([("/tmp/blob.bin", b"nope")])
print("to /tmp      :", rejected[0].error)

print("roundtrip    :", backend.download_files(["/mnt/data/blob.bin"])[0].content == bytes(range(256)))

to /mnt/data : ok
to /tmp      : invalid_path


roundtrip    : True


There is nothing to clean up here. Sessions are pool-managed and expire on their
own — which is exactly the trade we make next.

---

# Part 2 — Sandboxes

A **sandbox** (`Microsoft.App/sandboxGroups`) is a microVM you create, hold, and
delete. Where a session is a slot in a managed pool, a sandbox is a resource you
own:

- it **keeps state** — stop it, resume it later, snapshot it
- it has **real networking** — egress policies, VNet integration, exposed ports
- it can mount **volumes** — Azure Blob, Data Disk
- it **bills until you delete it**

That last point drives the shape of this section: everything runs inside a
`try/finally` so an interrupted run does not leave a microVM billing.

## 2.1 The two clients

The SDK splits control plane from data plane, and the split matters:

- **`SandboxGroupClient`** — the group. Creates, lists and deletes sandboxes;
  manages disk images, volumes, snapshots and secrets.
- **`SandboxClient`** — one sandbox. Executes commands, moves files, stops and
  resumes.

`ACASandbox` wraps a `SandboxClient`. It never builds an endpoint and never
handles a credential itself, so **you** keep control of the lifecycle through the
same client you passed in.

In [16]:
from azure.containerapps.sandbox import SandboxGroupClient, endpoint_for_region
from azure.identity import DefaultAzureCredential

group = SandboxGroupClient(
    endpoint_for_region(REGION),
    DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group=RESOURCE_GROUP,
    sandbox_group=SANDBOX_GROUP,
)

print("disk images:", sorted(image.name for image in group.list_public_disk_images()))

disk images: ['claude', 'copilot', 'dotnet-10', 'dotnet-8', 'dotnet-9', 'nginx', 'node-22', 'node-24', 'php-8.3', 'php-8.4', 'python-3.11', 'python-3.12', 'python-3.13', 'python-3.14', 'ubuntu']


Those are the presets. `python-3.12` gives us an interpreter and `pip`;
`ubuntu` is bare (and notably has `python3` but no `python`). You can also build
your own disk image, or bring an OCI image.

Before creating anything, note what is already in the group — so we only ever
delete our own:

In [17]:
pre_existing = {sandbox.id for sandbox in group.list_sandboxes()}
print(f"{len(pre_existing)} sandbox(es) already in the group; leaving them alone")

1 sandbox(es) already in the group; leaving them alone


## 2.2 Creating one

`begin_create_sandbox` is a long-running operation. Two details are easy to get
wrong, and both save you a step:

- `.result()` returns a **`SandboxClient`**, not a `Sandbox` model
- the poller has **already waited for `Running`** — no `ensure_running()` needed

Labels are free-form and are the practical way to find orphans later.

In [18]:
sandbox_client = group.begin_create_sandbox(
    disk="python-3.12",
    labels={"purpose": "deepagents-notebook"},
).result()

print("sandbox id :", sandbox_client.sandbox_id)
print("state      :", sandbox_client.get().state)

sandbox id : 64315e8c-ca65-4da7-8f8a-2160af043aa7
state      : Running


## 2.3 `ACASandbox`

One line. From here the object satisfies the same Deep Agents backend protocol
`SessionsBashBackend` did — the same `execute`/`read`/`write`/`glob`/`grep`
surface, backed by a microVM instead of a session.

In [19]:
from langchain_azure_container_apps.sandboxes import ACASandbox

sandbox = ACASandbox(sandbox_client)

print("id      :", sandbox.id)
print("python  :", sandbox.execute("python --version").output.strip())
print("cpus    :", sandbox.execute("nproc").output.strip())

id      : 64315e8c-ca65-4da7-8f8a-2160af043aa7


/var/folders/0w/mqr_p79n6n78wrvzd45x2jc40000gn/T/ipykernel_6182/1217270560.py:3: LangChainBetaWarning: `ACASandbox` is in public preview. Its API is not stable and may change in future versions.
  sandbox = ACASandbox(sandbox_client)


python  : Python 3.12.13


cpus    : 1


In [20]:
sandbox.write("/work/data.csv", "name,score\nada,91\ngrace,97\nalan,88\n")

read = sandbox.read("/work/data.csv")
print(read.file_data["content"])
print("total_lines:", read.total_lines)

# glob takes the pattern and the directory separately -- "/work/*.csv" as a
# single pattern globs the default directory and quietly matches nothing.
print("ls   :", [entry["path"] for entry in sandbox.ls("/work").entries])
print("glob :", [entry["path"] for entry in sandbox.glob("*.csv", path="/work").matches])
print("grep :", sandbox.grep("grace", path="/work").matches)

name,score
ada,91
grace,97
alan,88
total_lines: 4
ls   : ['/work/data.csv']


glob : ['data.csv']
grep : [{'path': '/work/data.csv', 'line': 3, 'text': 'grace,97'}]


Note the path convention, which is protocol behaviour rather than an ACA quirk
and trips people up: **`ls` returns absolute paths, `glob` returns paths relative
to the directory searched.** As in Python's `glob`, a wildcard will not match a
leading dot — name the hidden entry explicitly (`.env`, `.github/**`) to reach it.

Binary transfer goes through the SDK rather than the shell, so unlike the session
backend there is no `/mnt/data` restriction — any path works:

In [21]:
blob = bytes(range(256))
print("upload   :", sandbox.upload_files([("/work/bytes.bin", blob)])[0].error or "ok")
print("verified :", sandbox.download_files(["/work/bytes.bin"])[0].content == blob)

upload   : ok


verified : True


### Timeouts

`SandboxClient.exec()` has **no server-side command timeout**, so
`execute(..., timeout=N)` is honored by wrapping the command in coreutils
`timeout`. Exit code 124 is the standard signal:

In [22]:
timed_out = sandbox.execute("sleep 30", timeout=2)
print("exit_code :", timed_out.exit_code)
print("output    :", timed_out.output.strip())

exit_code : 124
output    : Command timed out (exit 124).


Two caveats on that. On a disk image without coreutils `timeout` the command
still runs — untimed. And the HTTP request itself is bounded by the transport's
read timeout (~300s on the SDK default): a command that outruns it fails the
*request* first, so `execute` returns an error response rather than raising, and
the command may keep running server-side. For timeouts above ~300s, construct
`SandboxClient` with a transport whose read timeout exceeds them.

## 2.4 The thing sessions cannot do: install something and keep it

The sandbox has network egress. Combined with persistent state, that means an
agent can build up a working environment over time instead of rebuilding it on
every call.

In [23]:
install = sandbox.execute("pip install --quiet humanize", timeout=120)
print("exit_code :", install.exit_code)
print("import    :", sandbox.execute(
    "python -c 'import humanize; print(humanize.intword(12_345_678))'"
).output.strip())

exit_code : 0


import    : 12.3 million


Now stop the sandbox. This is a real stop — the microVM is not running, and you
are not paying for its compute.

In [24]:
sandbox_client.stop()
print("state:", sandbox_client.get().state)

state: Stopped


In [25]:
sandbox_client.resume()
print("state:", sandbox_client.get().state)

print("file still there    :", sandbox.read("/work/data.csv").total_lines, "lines")
print("package still there :", sandbox.execute(
    "python -c 'import humanize; print(humanize.naturalsize(1048576))'"
).output.strip())

state: Running
file still there    : 4 lines


package still there : 1.0 MB


The file and the `pip install` both survived. **This is the whole reason to
choose sandboxes over sessions.** A session that goes idle takes everything with
it; a sandbox is a workspace an agent can return to across tasks, across
conversations, across days.

## 2.5 A Deep Agent on the sandbox

Same call as Part 1 — only the backend changed.

In [26]:
sandbox_agent = create_deep_agent(
    model=model,
    backend=sandbox,
    system_prompt=(
        "You are a data analyst working in a Linux sandbox with Python 3.12 "
        "and network access. You may pip install what you need. Write scripts "
        "to files, run them with execute, and report the real output. Keep "
        "command output small."
    ),
)

analysis = sandbox_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "/work/data.csv holds names and scores. Write "
                    "/work/analyse.py that reads it with the csv module and "
                    "prints the mean score and the top scorer. Run it and "
                    "report the output."
                ),
            }
        ]
    }
)
print(analysis["messages"][-1].content)

Done — `/work/analyse.py` was created and run.

Output:
```text
Mean score: 92.00
Top scorer: grace (97)
```


In [27]:
for message in analysis["messages"]:
    for call in getattr(message, "tool_calls", []) or []:
        args = call["args"]
        detail = args.get("file_path") or args.get("command") or ""
        print(f"{call['name']:12} {str(detail)[:70]}")

ls           
read_file    /work/data.csv
write_file   /work/analyse.py
execute      python3 /work/analyse.py


The agent's file lives in the sandbox, so we can read it back with the same
backend the agent used:

In [28]:
print(sandbox.read("/work/analyse.py").file_data["content"])

import csv
from statistics import mean

with open('/work/data.csv', newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

scores = [int(row['score']) for row in rows]
mean_score = mean(scores)
top = max(rows, key=lambda row: int(row['score']))

print(f"Mean score: {mean_score:.2f}")
print(f"Top scorer: {top['name']} ({top['score']})")


## 2.6 Snapshots

A snapshot captures the sandbox so a future one can start from it. The payoff for
agents is a **golden environment**: install and configure once, snapshot, then
launch every subsequent agent from that point instead of re-running `pip install`
on every cold start.

In [29]:
snapshot = sandbox_client.create_snapshot(name=f"nb-{sandbox_client.sandbox_id[:8]}")

# The model identifies a snapshot by `id`; there is no `name` attribute on the
# way back out, and delete_snapshot takes that id.
print("id        :", snapshot.id)
print("of sandbox:", snapshot.sandbox_id)
print("created   :", snapshot.created_at_utc)

id        : c67c58b9-04e3-4968-993f-5c640c45e048
of sandbox: 64315e8c-ca65-4da7-8f8a-2160af043aa7
created   : 2026-08-04T20:50:08.0095034Z


## 2.7 Teardown

**A sandbox bills until it is deleted.** Run this cell even if something above
failed — that is the point of keeping the id in a variable.

In [30]:
# Delete the snapshot first, then the sandbox, and never let a failure on one
# skip the other -- a raised exception here is exactly how a microVM gets
# orphaned and bills forever.
for label, delete in [
    ("snapshot", lambda: group.delete_snapshot(snapshot.id)),
    ("sandbox", lambda: group.delete_sandbox(sandbox_client.sandbox_id)),
]:
    try:
        delete()
        print(f"{label:8} deleted")
    except Exception as exc:  # noqa: BLE001 -- teardown must not stop at the first failure
        print(f"{label:8} FAILED: {exc}")

remaining = {s.id for s in group.list_sandboxes()}
print()
print("our sandbox gone        :", sandbox_client.sandbox_id not in remaining)
print("orphans we created      :", (remaining - pre_existing) or "none")
print("pre-existing, left alone:", len(remaining & pre_existing))

snapshot deleted


sandbox  deleted

our sandbox gone        : True
orphans we created      : none
pre-existing, left alone: 1


If a run dies partway and you lose the variable, the group is the source of
truth — list it and delete by label:

```python
for sandbox in group.list_sandboxes():
    print(sandbox.id, sandbox.state, sandbox.created_at)
```

For production, prefer a lifecycle policy over remembering:
`sandbox_client.set_lifecycle_policy(...)` lets the service reap idle sandboxes
for you.

---

# Part 3 — Choosing between them

Both integrations satisfy the same Deep Agents protocol, so switching is a
one-line change. The decision is about the resource underneath, not the code.

**Reach for dynamic sessions when** the unit of work is a single turn. Evaluating
model-written code, doing arithmetic, running a snippet against user data. No
lifecycle to manage, near-instant start, and the pool absorbs concurrency — one
endpoint serves as many `session_id`s as you throw at it. Accept in exchange:
ephemeral state, no persistent storage, a 4 KB output cap, and `/mnt/data`-only
file uploads.

**Reach for sandboxes when** the agent needs a workspace. A long-running coding
agent, a dev environment, anything that installs dependencies or accumulates
artifacts across turns. You get stop/resume, snapshots, volumes, egress control
and unrestricted file paths. Accept in exchange: you own the lifecycle, it bills
until deleted, and each one takes ~10s to provision.

A useful test: **if losing all state between calls would be a bug, you want a
sandbox.**

### Patterns worth stealing

- **Per-user isolation.** Sessions: derive `session_id` from your user or thread
  id. Sandboxes: one sandbox per user, labelled, reaped by lifecycle policy.
- **Golden snapshots.** Provision, install, snapshot once; start every agent from
  the snapshot rather than paying setup cost per cold start.
- **Always `try/finally`.** A sandbox leaked by an exception bills indefinitely.

### Where to look next

- Package README — `libs/azure-container-apps/README.md`
- [Deep Agents docs](https://docs.langchain.com/labs/deep-agents/overview)
- [ACA dynamic sessions](https://learn.microsoft.com/azure/container-apps/sessions)
- [ACA sandboxes](https://learn.microsoft.com/azure/container-apps/)

`langchain-azure-dynamic-sessions` is the older, published package. It is
deprecated in favour of this one, which additionally carries the Deep Agents
backends and fixes the dynamic-sessions bugs that the old package still has.